# Events missing in `withoutcoinc/`

Compare the topology HDF5 files in:
- `Bfr/` — full event selection (including coincident events)
- `Bfr/withoutcoinc/` — event selection with coincident events removed

and find which events are present in `Bfr/` but absent in `withoutcoinc/`.

In [ ]:
import tables
import pandas as pd
import numpy as np

In [ ]:
BFR = "/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/hdf_files/NoDeepCore/HESE12/Bfr"
WOC = "/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/hdf_files/NoDeepCore/HESE12/Bfr/withoutcoinc"

# withoutcoinc/ has no combined HESE12.hdf5, only the topology files
TOPOLOGIES = ["Cascades", "DoubleCascades", "Tracks"]

## Per-topology comparison

In [ ]:
def load_df(path):
    with tables.open_file(path, "r") as f:
        return pd.DataFrame({
            "Run":       f.root.I3EventHeader.col("Run"),
            "Event":     f.root.I3EventHeader.col("Event"),
            "RecoETot":  f.root.RecoETot.col("value"),
            "QTot":      f.root.QTot.col("value"),
        })

missing_all = []

for topo in TOPOLOGIES:
    fname = f"HESE12_{topo}.hdf5"
    df_bfr = load_df(f"{BFR}/{fname}")
    df_woc = load_df(f"{WOC}/{fname}")

    set_bfr = set(zip(df_bfr["Run"], df_bfr["Event"]))
    set_woc = set(zip(df_woc["Run"], df_woc["Event"]))

    only_bfr = set_bfr - set_woc
    only_woc = set_woc - set_bfr

    print(f"{topo}: Bfr={len(df_bfr)}, withoutcoinc={len(df_woc)}, "
          f"only_bfr={len(only_bfr)}, only_woc={len(only_woc)}")

    if only_bfr:
        rows = df_bfr[[(r, e) in only_bfr for r, e in zip(df_bfr["Run"], df_bfr["Event"])]].copy()
        rows.insert(0, "Topology", topo)
        missing_all.append(rows)

df_missing = pd.concat(missing_all, ignore_index=True) if missing_all else pd.DataFrame()

## Events in `Bfr/` but missing in `withoutcoinc/`

In [ ]:
if df_missing.empty:
    print("No missing events.")
else:
    print(f"{len(df_missing)} event(s) present in Bfr/ but absent in withoutcoinc/:")
    display(df_missing[["Topology", "Run", "Event", "RecoETot", "QTot"]].reset_index(drop=True))